# Assignment 2 — Prompt Engineering & Few-Shot Prediction
**Course:** Large Language Models for Marketing (FEM11154)

This notebook covers:
1. **Data Setup** — creates shared sample files for all group members
2. **Task 2** — Zero / One / Few-shot prompting to predict Sephora review star ratings


---
## STEP 0 — Install & Import Libraries

In [1]:
# Install OpenAI library
!pip install openai -q

import pandas as pd
import numpy as np
from openai import OpenAI
from sklearn.metrics import mean_absolute_error, accuracy_score
import time
import re

print('All libraries loaded successfully!')

---
## STEP 1 — Load Dataset
Upload `sephora_sample_10k.csv` using the file icon on the left sidebar in Colab, then run this cell.

In [2]:
# Load the full dataset
df = pd.read_csv('sephora_sample_10k.csv')

# Drop rows with missing review text or rating
df = df.dropna(subset=['review_text', 'rating']).reset_index(drop=True)

print(f'Dataset loaded: {len(df)} rows')
print(f'Rating distribution:')
print(df['rating'].value_counts().sort_index())

Dataset loaded: 10000 rows
Rating distribution:
rating
1     518
2     474
3     748
4    1874
5    6386
Name: count, dtype: int64


---
## STEP 2 — Data Setup (Shared Files for All Group Members)
This creates 4 CSV files to share with your group. Run this once and send the files.

In [6]:
# ── RANDOM SEED (everyone must use this same seed) ──────────────────────────
SEED = 42

# ── PERSON 1: 15 reviews (3 per star rating) ────────────────────────────────
person1_sample = df.groupby('rating', group_keys=False).apply(
    lambda x: x.sample(3, random_state=SEED)
).reset_index(drop=True)
person1_sample.to_csv('sample_person1.csv', index=False)
print(f'Person 1 sample: {len(person1_sample)} reviews saved to sample_person1.csv')

# ── SHARED TEST SET: 300 reviews (60 per star rating) ────────────────────────
# Remove person1 reviews first to avoid overlap
df_remaining = df.drop(person1_sample.index).reset_index(drop=True)

shared_test = df_remaining.groupby('rating', group_keys=False).apply(
    lambda x: x.sample(60, random_state=SEED)
).reset_index(drop=True)
shared_test.to_csv('shared_test_set.csv', index=False)
print(f'Shared test set: {len(shared_test)} reviews saved to shared_test_set.csv')

# ── PERSON 3 TRAINING SET: 1000 reviews (not overlapping with test set) ──────
df_remaining2 = df_remaining.drop(shared_test.index).reset_index(drop=True)

person3_train = df_remaining2.groupby('rating', group_keys=False).apply(
    lambda x: x.sample(min(200, len(x)), random_state=SEED)
).reset_index(drop=True).sample(1000, random_state=SEED).reset_index(drop=True)
person3_train.to_csv('person3_train.csv', index=False)
print(f'Person 3 train set: {len(person3_train)} reviews saved to person3_train.csv')

# ── ONE-SHOT EXAMPLES (picked directly from full dataset) ────────────────────
# Pick a clear 5-star example
example_positive = df[df['rating'] == 5].copy()
example_positive['length'] = example_positive['review_text'].str.len()
example_positive = example_positive[example_positive['length'].between(100, 400)].iloc[0]

# Pick a clear 1-star example
example_negative = df[df['rating'] == 1].copy()
example_negative['length'] = example_negative['review_text'].str.len()
example_negative = example_negative[example_negative['length'].between(100, 400)].iloc[0]

print(f'\nPositive example (5 stars):')
print(example_positive['review_text'][:200])
print(f'\nNegative example (1 star):')
print(example_negative['review_text'][:200])

print('\nAll data files created successfully!')

Person 1 sample: 15 reviews saved to sample_person1.csv
Shared test set: 300 reviews saved to shared_test_set.csv
Person 3 train set: 1000 reviews saved to person3_train.csv

Positive example (5 stars):
I have combination skin and I was previously using a drugstore cleanser with my Foreo. I switched it up and started using the Soy Face Cleanser alone - I have had no pimples or irritation and some of 

Negative example (1 star):
This new formula is horrible. Doesn’t matter how much I shake it, it still comes out almost chunky and it doesn’t spread well on my skin. I have also been breaking out ever since the formula changed. 

All data files created successfully!


---
## STEP 3 — Set Up OpenAI API
Paste your API key below. It stays private in your own Colab notebook.

In [7]:
# ── PASTE YOUR API KEY HERE ──────────────────────────────────────────────────
API_KEY = ' '

client = OpenAI(api_key=API_KEY)

# Quick test to confirm the key works
test = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': 'Reply with the number 5 only.'}],
    max_tokens=5
)
print('API key works! Test response:', test.choices[0].message.content)

API key works! Test response: 5


---
## STEP 4 — Define the 4 Prompt Conditions

In [8]:
# ── SYSTEM PROMPT (same for all conditions) ──────────────────────────────────
SYSTEM_PROMPT = """You are a review rating predictor for a beauty products retailer.
Your task is to read a customer review and predict the star rating from 1 to 5.
Reply with a single number only: 1, 2, 3, 4, or 5. Do not add any explanation."""

# ── FEW-SHOT EXAMPLES (one per star rating) ───────────────────────────────────
# We use 5 fixed examples (one per rating) for the few-shot condition
few_shot_examples = df_remaining2.groupby('rating', group_keys=False).apply(
    lambda x: x.sample(1, random_state=99)
).reset_index(drop=True)

def build_few_shot_block():
    """Build the few-shot examples text block."""
    block = ''
    for _, row in few_shot_examples.iterrows():
        block += f'Review: {row["review_text"][:300]}\nRating: {int(row["rating"])}\n\n'
    return block

FEW_SHOT_BLOCK = build_few_shot_block()

# ── PROMPT BUILDER FUNCTIONS ──────────────────────────────────────────────────
def make_zero_shot_prompt(review):
    """No examples — just the review."""
    return f'Review: {review}\nRating:'

def make_one_shot_pos_prompt(review):
    """One positive (5-star) example before the review."""
    return (
        f'Here is an example:\n'
        f'Review: {example_positive["review_text"][:300]}\n'
        f'Rating: 5\n\n'
        f'Now predict this one:\n'
        f'Review: {review}\nRating:'
    )

def make_one_shot_neg_prompt(review):
    """One negative (1-star) example before the review."""
    return (
        f'Here is an example:\n'
        f'Review: {example_negative["review_text"][:300]}\n'
        f'Rating: 1\n\n'
        f'Now predict this one:\n'
        f'Review: {review}\nRating:'
    )

def make_few_shot_prompt(review):
    """Five examples (one per star rating) before the review."""
    return (
        f'Here are some examples:\n\n'
        f'{FEW_SHOT_BLOCK}'
        f'Now predict this one:\n'
        f'Review: {review}\nRating:'
    )

CONDITIONS = {
    'zero_shot':    make_zero_shot_prompt,
    'one_shot_pos': make_one_shot_pos_prompt,
    'one_shot_neg': make_one_shot_neg_prompt,
    'few_shot':     make_few_shot_prompt,
}

print('Prompt conditions defined:')
for name in CONDITIONS:
    print(f'  - {name}')

Prompt conditions defined:
  - zero_shot
  - one_shot_pos
  - one_shot_neg
  - few_shot


---
## STEP 5 — Run Predictions
This cell calls the OpenAI API for all 300 reviews × 4 conditions = 1,200 calls.

⏱️ **Estimated time: 10–20 minutes.** A progress counter will update every 50 rows.

In [9]:
def call_api(system_prompt, user_prompt, retries=3):
    """Call OpenAI API with retry logic on failure."""
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model='gpt-4o-mini',
                messages=[
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user',   'content': user_prompt}
                ],
                max_tokens=5,
                temperature=0  # deterministic outputs for reproducibility
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            print(f'API error (attempt {attempt+1}): {e}')
            time.sleep(5)
    return None

def extract_rating(text):
    """Extract a number 1-5 from the model's response."""
    if text is None:
        return None
    match = re.search(r'[1-5]', text)
    return int(match.group()) if match else None

# ── RUN ALL CONDITIONS ────────────────────────────────────────────────────────
results = []
total = len(shared_test) * len(CONDITIONS)
counter = 0

for condition_name, prompt_fn in CONDITIONS.items():
    print(f'\nRunning condition: {condition_name}...')

    for i, row in shared_test.iterrows():
        review_text = str(row['review_text'])[:500]  # truncate very long reviews
        true_rating = int(row['rating'])

        user_prompt = prompt_fn(review_text)
        raw_output  = call_api(SYSTEM_PROMPT, user_prompt)
        predicted   = extract_rating(raw_output)

        results.append({
            'condition':        condition_name,
            'review_text':      review_text,
            'true_rating':      true_rating,
            'raw_output':       raw_output,
            'predicted_rating': predicted
        })

        counter += 1
        if counter % 50 == 0:
            print(f'  Progress: {counter}/{total} calls done...')

        # Small pause to avoid hitting rate limits
        time.sleep(0.1)

# Save results
results_df = pd.DataFrame(results)
results_df.to_csv('person2_results.csv', index=False)
print(f'\nDone! {len(results_df)} predictions saved to person2_results.csv')


Running condition: zero_shot...
  Progress: 50/1200 calls done...
  Progress: 100/1200 calls done...
  Progress: 150/1200 calls done...
  Progress: 200/1200 calls done...
  Progress: 250/1200 calls done...
  Progress: 300/1200 calls done...

Running condition: one_shot_pos...
  Progress: 350/1200 calls done...
  Progress: 400/1200 calls done...
  Progress: 450/1200 calls done...
  Progress: 500/1200 calls done...
  Progress: 550/1200 calls done...
  Progress: 600/1200 calls done...

Running condition: one_shot_neg...
  Progress: 650/1200 calls done...
  Progress: 700/1200 calls done...
  Progress: 750/1200 calls done...
  Progress: 800/1200 calls done...
  Progress: 850/1200 calls done...
  Progress: 900/1200 calls done...

Running condition: few_shot...
  Progress: 950/1200 calls done...
  Progress: 1000/1200 calls done...
  Progress: 1050/1200 calls done...
  Progress: 1100/1200 calls done...
  Progress: 1150/1200 calls done...
  Progress: 1200/1200 calls done...

Done! 1200 predict

---
## STEP 6 — Evaluate Results

In [10]:
# Load results (run this even if you restart the notebook)
results_df = pd.read_csv('person2_results.csv')

print('=== RESULTS TABLE ===')
print(f'{"Condition":<20} {"Accuracy":>10} {"MAE":>8} {"N predictions":>15}')
print('-' * 60)

summary_rows = []
for condition in ['zero_shot', 'one_shot_pos', 'one_shot_neg', 'few_shot']:
    subset = results_df[
        (results_df['condition'] == condition) &
        (results_df['predicted_rating'].notna())
    ]

    acc = accuracy_score(subset['true_rating'], subset['predicted_rating'])
    mae = mean_absolute_error(subset['true_rating'], subset['predicted_rating'])
    n   = len(subset)

    print(f'{condition:<20} {acc:>9.1%} {mae:>8.3f} {n:>15}')
    summary_rows.append({'Condition': condition, 'Accuracy': round(acc, 4), 'MAE': round(mae, 3), 'N': n})

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv('person2_summary.csv', index=False)
print('\nSummary saved to person2_summary.csv')

=== RESULTS TABLE ===
Condition              Accuracy      MAE   N predictions
------------------------------------------------------------
zero_shot                65.3%    0.357             300
one_shot_pos             67.0%    0.340             300
one_shot_neg             63.7%    0.377             300
few_shot                 63.3%    0.380             300

Summary saved to person2_summary.csv


---
## STEP 7 — Deeper Analysis

In [11]:
# ── 1. Average predicted rating per condition (bias check) ───────────────────
print('=== AVERAGE PREDICTED RATING PER CONDITION ===')
print('(True average rating in test set:', round(results_df['true_rating'].mean(), 2), ')')
print()

avg_pred = results_df.groupby('condition')['predicted_rating'].mean().round(3)
print(avg_pred.to_string())

print()
print('>>> If one_shot_pos predicts higher on average than one_shot_neg,')
print('    this confirms the example biases the model.')

=== AVERAGE PREDICTED RATING PER CONDITION ===
(True average rating in test set: 3.0 )

condition
few_shot        3.073
one_shot_neg    3.063
one_shot_pos    3.027
zero_shot       3.043

>>> If one_shot_pos predicts higher on average than one_shot_neg,
    this confirms the example biases the model.


In [12]:
# ── 2. Accuracy broken down by true star rating ───────────────────────────────
print('=== ACCURACY PER STAR RATING (few_shot condition) ===')
few = results_df[
    (results_df['condition'] == 'few_shot') &
    (results_df['predicted_rating'].notna())
].copy()

for star in [1, 2, 3, 4, 5]:
    sub = few[few['true_rating'] == star]
    if len(sub) > 0:
        acc = accuracy_score(sub['true_rating'], sub['predicted_rating'])
        print(f'  Star {star}: {acc:.1%} accuracy ({len(sub)} reviews)')

=== ACCURACY PER STAR RATING (few_shot condition) ===
  Star 1: 60.0% accuracy (60 reviews)
  Star 2: 53.3% accuracy (60 reviews)
  Star 3: 55.0% accuracy (60 reviews)
  Star 4: 56.7% accuracy (60 reviews)
  Star 5: 91.7% accuracy (60 reviews)


In [13]:
# ── 3. Confusion between one_shot_pos vs one_shot_neg ────────────────────────
print('=== ONE-SHOT POSITIVE vs NEGATIVE: RATING DISTRIBUTION ===')
for cond in ['one_shot_pos', 'one_shot_neg']:
    sub = results_df[results_df['condition'] == cond]['predicted_rating']
    dist = sub.value_counts().sort_index()
    print(f'\n{cond}:')
    for rating, count in dist.items():
        bar = '█' * int(count / 3)
        print(f'  {int(rating)} stars: {count:>3}  {bar}')

=== ONE-SHOT POSITIVE vs NEGATIVE: RATING DISTRIBUTION ===

one_shot_pos:
  1 stars:  47  ███████████████
  2 stars:  84  ████████████████████████████
  3 stars:  49  ████████████████
  4 stars:  54  ██████████████████
  5 stars:  66  ██████████████████████

one_shot_neg:
  1 stars:  44  ██████████████
  2 stars:  81  ███████████████████████████
  3 stars:  58  ███████████████████
  4 stars:  46  ███████████████
  5 stars:  71  ███████████████████████


---
## STEP 8 — Key Findings Template
Use this as a starting point for your report section.

In [14]:
# Print a ready-made findings summary to copy into your report
print('=== KEY FINDINGS SUMMARY FOR REPORT ===')
print()

for condition in ['zero_shot', 'one_shot_pos', 'one_shot_neg', 'few_shot']:
    subset = results_df[
        (results_df['condition'] == condition) &
        (results_df['predicted_rating'].notna())
    ]
    acc = accuracy_score(subset['true_rating'], subset['predicted_rating'])
    mae = mean_absolute_error(subset['true_rating'], subset['predicted_rating'])
    avg = subset['predicted_rating'].mean()
    print(f'{condition}: Accuracy={acc:.1%}, MAE={mae:.3f}, Avg predicted rating={avg:.2f}')

print()
pos_avg = results_df[results_df['condition']=='one_shot_pos']['predicted_rating'].mean()
neg_avg = results_df[results_df['condition']=='one_shot_neg']['predicted_rating'].mean()
bias    = round(pos_avg - neg_avg, 3)
print(f'Bias from example: one_shot_pos avg={pos_avg:.2f}, one_shot_neg avg={neg_avg:.2f}')
print(f'Difference = {bias} (positive = positive example pushes ratings higher)')

=== KEY FINDINGS SUMMARY FOR REPORT ===

zero_shot: Accuracy=65.3%, MAE=0.357, Avg predicted rating=3.04
one_shot_pos: Accuracy=67.0%, MAE=0.340, Avg predicted rating=3.03
one_shot_neg: Accuracy=63.7%, MAE=0.377, Avg predicted rating=3.06
few_shot: Accuracy=63.3%, MAE=0.380, Avg predicted rating=3.07

Bias from example: one_shot_pos avg=3.03, one_shot_neg avg=3.06
Difference = -0.037 (positive = positive example pushes ratings higher)
